# Module 2 · Lab 2.1 — LangChain Fundamentals: Chains, Memory & LCEL Composition

**Scenario.** Customer-complaint pipeline (sentiment + categorization + reply drafting). The content is illustrative on purpose — the focus is LCEL mechanics, not the customer-support business.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Compose LCEL chains with `prompt | llm | parser` and stream their output.
2. Give a chain conversational memory with a **per-session message store** — and explain why the `ConversationBufferMemory` family and `RunnableWithMessageHistory`'s history classes are both deprecated.
3. Emit granular streaming events with **`.astream_events()`** for streaming UIs (the foundation for streaming UIs).
4. Compose multi-step pipelines with `RunnableLambda` and `RunnablePassthrough`.
5. Run independent chain branches in parallel with **`RunnableParallel`** and measure the latency win.

---


> 🗺️ **Workshop run-list.**
> **Live in session:** all sections — §4 LCEL chains (compose, JSON, stream), §5 conversational memory, §6 streaming events, §7 multi-step composition, §8 parallel branches ⭐, §9

## 1. Install Dependencies

In [1]:
# 📦 Packages this notebook needs. On the workshop VM they are pre-installed from
# the workshop's requirements.txt, so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU "langchain>=1.0" "langchain-core>=1.0" "langchain-openai>=1.0" "langchain-groq>=0.2" pydantic

from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_PACKAGES = [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langchain-groq",
    "pydantic",
]
for package_name in NOTEBOOK_PACKAGES:
    try:
        print(f"{package_name:<26} {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:<26} NOT INSTALLED — uncomment the !pip line above")

langchain                  1.4.3
langchain-core             1.6.6
langchain-openai           1.6.6
langchain-groq             1.1.3
pydantic                   2.12.5


## 2. API Key Configuration

In [2]:
import os

# Where to create each key. The trainer supplies OPENAI_API_KEY on the workshop VM;
# you create any other key yourself and add it to the workshop's .env file as KEY_NAME=value.
KEY_SIGNUP_PAGES = {
    "OPENAI_API_KEY": "supplied on the workshop VM (elsewhere: https://platform.openai.com/api-keys)",
    "TAVILY_API_KEY": "https://app.tavily.com (free tier)",
    "GROQ_API_KEY": "https://console.groq.com/keys (free tier)",
    "WEATHER_API_KEY": "https://www.weatherapi.com/signup.aspx (free tier)",
    "LANGSMITH_API_KEY": "https://smith.langchain.com -> Settings -> API Keys (free tier)",
    "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
    "GEMINI_API_KEY": "https://aistudio.google.com/app/apikey",
}


def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere).

    Prints one line per key, then stops with instructions if a required key is missing.

    Args:
        required: the environment-variable names this lab needs.
    """
    try:
        from google.colab import userdata          # running on Google Colab
        for key_name in required:
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # the workshop VM, or any local machine
        from dotenv import find_dotenv, load_dotenv
        load_dotenv(find_dotenv(usecwd=True))        # searches upward for the workshop's .env file
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for key_name in required:
        print(f"   {key_name:<22} {'✅' if os.environ.get(key_name) else '❌ MISSING'}")
    missing_keys = [key_name for key_name in required if not os.environ.get(key_name)]
    if missing_keys:
        instructions = "\n".join(
            f"  {key_name}: {KEY_SIGNUP_PAGES.get(key_name, 'see README.md in the workshop folder')}"
            for key_name in missing_keys)
        raise RuntimeError(
            "This lab needs API keys that are not set yet. Create each one, add it to\n"
            "the workshop's .env file as KEY_NAME=value, then re-run this cell:\n" + instructions)


load_keys(["OPENAI_API_KEY", "GROQ_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅
   GROQ_API_KEY           ✅


## 3. Initialize LangChain LLM Wrappers

We'll use OpenAI and Groq side-by-side throughout this module — the exact same chain works on both, demonstrating the provider-agnostic value of LangChain abstractions.


In [3]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Primary model — OpenAI gpt-4.1-mini (fast, cheap, supports temperature)
llm_openai = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)

# Open-weight alternative — Qwen3.8 27B via Groq's LPU hardware
# (Groq no longer serves Llama 3.3 70B; Qwen3.8 27B is the open-weight model used in its place)
llm_groq = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.7)

# Quick smoke test — both should respond to a simple invoke
print("OpenAI:", llm_openai.invoke("Say 'hello' in one word.").content)
print("Groq:  ", llm_groq.invoke("Say 'hello' in one word.").content)
print("\nBoth LLM clients ready ✅")

OpenAI: Hello.
Groq:   Hello

Both LLM clients ready ✅


---

## 4. Basic LCEL Chains — Compose, Stream, Parse

The LCEL pipe (`|`) is sugar for `RunnableSequence`. Three runnables in a row → three steps in your pipeline. Stream the output, swap providers, replace the parser — each is a one-line change.


### 4a. Basic Chain with `StrOutputParser`

**Pattern:** `prompt | llm | parser`. The prompt template is rendered with input variables, the LLM produces a `BaseMessage`, and the parser extracts the string content.


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt template with placeholders
prompt = ChatPromptTemplate.from_template(
    """You are a marketing copywriter for a retail bank.
Write a concise 3-sentence product summary for the following loan product.

Product: {product_name}
Target Audience: {audience}

Summary:"""
)

# Build the chain: prompt → LLM → string parser
chain = prompt | llm_openai | StrOutputParser()

# Invoke with input variables
result = chain.invoke({
    "product_name": "SmartHome Loan",
    "audience": "First-time homebuyers aged 25-35"
})

print("--- OpenAI Result ---")
print(result)
print(f"\nType: {type(result)}")  # Should be <class 'str'>

--- OpenAI Result ---
The SmartHome Loan is designed to help first-time homebuyers aged 25-35 step confidently into homeownership with low interest rates and flexible repayment options. Enjoy personalized support and quick approval to make your dream home a reality without the hassle. Start building your future today with a loan tailored to your lifestyle and goals.

Type: <class 'langchain_core.messages.base.TextAccessor'>


In [5]:
# Same chain, swap to Groq — only the LLM changes, everything else stays the same
chain_groq = prompt | llm_groq | StrOutputParser()

result_groq = chain_groq.invoke({
    "product_name": "SmartHome Loan",
    "audience": "First-time homebuyers aged 25-35"
})

print("--- Groq (Qwen3.8 27B) Result ---")
print(result_groq)

--- Groq (Qwen3.8 27B) Result ---
The SmartHome Loan is designed specifically for first-time buyers aged 25 to 35, offering competitive rates and flexible terms to make your path to homeownership smoother. We simplify the complex mortgage process with low closing costs and streamlined digital application tools tailored for young professionals. Start building your future today with a loan that works as hard as you do.


### 4b. Structured Output with `JsonOutputParser`

When the response should be a JSON object (dict). Pair with `format_instructions` so the model knows the expected shape. Streamable and dict-typed.


In [6]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser without a Pydantic model — returns a plain dict
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_template(
    """Extract structured policy details from the following text.
Return a JSON object with keys: policy_type, coverage_amount, premium_monthly, deductible, term_years.

{format_instructions}

Text: {policy_text}"""
)

json_chain = json_prompt | llm_openai | json_parser

result = json_chain.invoke({
    "policy_text": """Our Gold Health Shield plan offers comprehensive coverage of up to $500,000
    with a monthly premium of $350 and an annual deductible of $1,500. The policy term is 5 years
    with guaranteed renewal.""",
    "format_instructions": json_parser.get_format_instructions()
})

print("Parsed JSON output:")
print(result)
print(f"\nType: {type(result)}")  # Should be <class 'dict'>
print(f"Coverage: {result.get('coverage_amount')}")

Parsed JSON output:
{'policy_type': 'Gold Health Shield', 'coverage_amount': 500000, 'premium_monthly': 350, 'deductible': 1500, 'term_years': 5}

Type: <class 'dict'>
Coverage: 500000


### 4c. Type-Safe Output (covered in Lab 1.1)

> Structured output with Pydantic is **Lab 1.1 §9**: `chat_model.with_structured_output(Model)` returns a validated object instead of text. We don't repeat it here.


### 4d. Streaming with LCEL

Calling `.stream(...)` instead of `.invoke(...)` yields tokens as they arrive. Same chain, different terminal method.


In [7]:
# Reuse the basic chain from 4a — streaming just works
stream_chain = prompt | llm_openai | StrOutputParser()

print("Streaming response token by token:\n")
for chunk in stream_chain.stream({
    "product_name": "Green Auto Loan",
    "audience": "Eco-conscious consumers buying electric vehicles"
    }):
    print(chunk, end="", flush=True)

print("\n\n✅ Streaming complete")

Streaming response token by token:



Drive

 green

 and

 save

...

 dream

 EV

 today

.



✅ Streaming complete


### Key Takeaways — Task 1

| Component | Class | What It Does |
|-----------|-------|--------------|
| **Prompt Template** | `ChatPromptTemplate` | Structures the input with placeholders |
| **LLM** | `ChatOpenAI` / `ChatGroq` | Generates the response |
| **StrOutputParser** | `StrOutputParser` | Returns plain text string |
| **JsonOutputParser** | `JsonOutputParser` | Parses into Python dict |
| **PydanticOutputParser** | `PydanticOutputParser` | Parses into validated Pydantic object |

The LCEL pipe (`|`) operator connects these components — output of one feeds into the next.

---

## 5. Conversational Memory — a per-session message store

A chat model is **stateless**: every call starts from nothing. "Memory" is simply the earlier messages
of the conversation, sent back in with the new one — and `MessagesPlaceholder` is the slot they go into.

**The deprecated-memory trap — twice over.**
- The old `ConversationBufferMemory` / `ConversationSummaryMemory` / `ConversationEntityMemory` classes
  are long deprecated. They lived outside LCEL (no `.invoke` / `.stream` / `.batch`), coupled history
  *storage* with history *injection*, and did not compose with `RunnableParallel`, fallbacks or retries.
- Their v1 replacement, `RunnableWithMessageHistory`, needs a history object — and the classes it relies
  on (`InMemoryChatMessageHistory`, `BaseChatMessageHistory`) were **deprecated in langchain-core 1.6.4**.

The idea that survives both is small, so we write it ourselves: **one list of messages per session,
injected through `MessagesPlaceholder`.** Nothing is hidden, and it composes with every LCEL feature.
Lab 4.2 scales this up — per-user threads, compression, persistence — and Lab 7.2 shows the
graph-side equivalent, LangGraph checkpointers.


In [8]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1) The underlying chain — accepts {input, history}
memory_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a friendly assistant. Remember what the user tells you.'),
    MessagesPlaceholder('history'),
    ('user', '{input}'),
])
memory_chain = memory_prompt | llm_openai

# 2) Per-session history store — one plain list of messages per session_id
conversation_store: dict[str, list] = {}


def get_history(session_id: str) -> list:
    """Return the message list for one session, creating an empty one on first use."""
    return conversation_store.setdefault(session_id, [])


# 3) One conversational turn: inject this session's history, call the chain, record both sides
def chat_with_memory(session_id: str, user_input: str):
    """Answer `user_input` in the context of this session's history, then store the exchange.

    Args:
        session_id: which conversation this turn belongs to.
        user_input: the user's new message.

    Returns:
        The model's reply (an AIMessage).
    """
    history = get_history(session_id)
    reply = memory_chain.invoke({'input': user_input, 'history': history})
    history.extend([HumanMessage(content=user_input), reply])
    return reply

In [9]:
# 3-turn conversation — the model should remember 'Priya' across turns
session_id = 'user-priya-001'

first_reply = chat_with_memory(session_id, 'Hi, my name is Priya and I like banking software.')
print('Turn 1 →', first_reply.content)

second_reply = chat_with_memory(session_id, 'What is my name?')
print('Turn 2 →', second_reply.content)

third_reply = chat_with_memory(session_id, 'Recommend a Python library I should learn given my interests.')
print('Turn 3 →', third_reply.content)

# A different session starts from nothing — history never leaks between sessions
other_reply = chat_with_memory('user-rahul-002', 'What is my name?')
print('\nNew session →', other_reply.content)

Turn 1 → Hi Priya! It's great to meet you. Banking software is a fascinating field. How can I assist you today?


Turn 2 → Your name is Priya. How can I help you further?


Turn 3 → Since you like banking software, I recommend learning **QuantLib**. It's a powerful Python library for quantitative finance, including pricing, risk management, and financial instrument modeling, which are essential in banking software development. 

Would you like me to suggest some resources to get started with QuantLib?



New session → I don't believe you've told me your name yet. What is your name?


In [10]:
# Inspect the history store — actual messages live here
history_messages = conversation_store['user-priya-001']
print(f'Total messages stored: {len(history_messages)}\n')
for position, message in enumerate(history_messages, 1):
    print(f'{position}. [{type(message).__name__}] {message.content[:80]}...')

Total messages stored: 6

1. [HumanMessage] Hi, my name is Priya and I like banking software....
2. [AIMessage] Hi Priya! It's great to meet you. Banking software is a fascinating field. How c...
3. [HumanMessage] What is my name?...
4. [AIMessage] Your name is Priya. How can I help you further?...
5. [HumanMessage] Recommend a Python library I should learn given my interests....
6. [AIMessage] Since you like banking software, I recommend learning **QuantLib**. It's a power...


### 📝 Key Takeaways — Section 5
- **Memory is data, not a class.** A list of messages per session, injected through
  `MessagesPlaceholder` — the chain itself stays stateless.
- **The `session_id` keeps conversations apart.** The same chain serves any number of parallel
  conversations without leaking state between them — the new session above did not know Priya.
- **Swap the storage, keep the chain.** For production the dictionary becomes a database, keyed by the
  same `session_id`; Lab 4.2 saves conversations to JSON and resumes them.
- **Agents and graphs** keep this state in a checkpointer instead (Lab 7.2).

---

## 6. Streaming Events with `.astream_events()`

`.stream(...)` yields the final tokens. `.astream_events(...)` yields **every internal event** in the chain — model start/end, parser events, tool starts, retry events. Foundation for streaming UIs that show "thinking", "calling tool X", or per-step progress.

It is what a streaming web endpoint — FastAPI with Server-Sent Events, for example — is built on.


In [11]:
import asyncio

# Use the basic chain from §4a — any LCEL chain works
stream_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise banking copywriter.'),
    ('user', '{topic}'),
])
stream_chain = stream_prompt | llm_openai

async def consume_events(topic: str) -> None:
    """Stream one chain run and print only the model start, token and end events."""
    async for event in stream_chain.astream_events({'topic': topic}, version='v2'):
        kind = event['event']
        # Show only the events most useful for a UI
        if kind == 'on_chat_model_start':
            print(f'[start] model={event["name"]}')
        elif kind == 'on_chat_model_stream':
            chunk = event['data']['chunk']
            if chunk.content:
                print(chunk.content, end='', flush=True)
        elif kind == 'on_chat_model_end':
            print('\n[end]')

await consume_events('Write a 2-sentence tagline for a UPI payments app called PayZap.')


[start] model=ChatOpenAI


Pay

Zap

:

 Fast

...

 anytime

,

 anywhere

.


[end]

In [12]:
# Inspecting all event types — useful for understanding what's available
seen_events = set()
async for event in stream_chain.astream_events({'topic': 'Write a haiku about banking.'}, version='v2'):
    seen_events.add(event['event'])

print('Event types emitted:')
for event_type in sorted(seen_events):
    print(f'  • {event_type}')


Event types emitted:
  • on_chain_end
  • on_chain_start
  • on_chain_stream
  • on_chat_model_end
  • on_chat_model_start
  • on_chat_model_stream
  • on_prompt_end
  • on_prompt_start


### 📝 When to use which streaming method

| Method | Returns | Use when |
|---|---|---|
| `.invoke()` | Final result, blocking | Backend services, batch pipelines, agents |
| `.stream()` | Final-output tokens, one chunk at a time | User-facing chat where you want to show the answer streaming in |
| `.astream_events()` | All internal events (start, stream, end, tool calls, etc.) | Streaming UIs that show *intermediate* state — "thinking", "calling tool X", per-step progress |

A FastAPI endpoint would use `.astream_events()` to power a Server-Sent-Events (SSE) stream.


---

## 7. Multi-Step Composition with `RunnableLambda`

Real pipelines have post-processing between steps — extracting a field, calling a deterministic function, reformatting. `RunnableLambda` is the escape hatch from LCEL into ordinary Python without breaking the pipe syntax.


In [13]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda

# --- Step 1: Extract structured details from raw complaint ---
extract_prompt = ChatPromptTemplate.from_template(
    """Extract key details from this customer complaint. Return ONLY valid JSON with these keys:
- customer_name
- issue_type (one of: billing, claim_denial, service_delay, coverage_question, other)
- severity (one of: low, medium, high, critical)
- product_mentioned
- brief_summary (one sentence)

Complaint: {complaint}"""
)

extract_chain = extract_prompt | llm_openai | JsonOutputParser()

# --- Step 2: Transform into internal ticket format ---
transform_prompt = ChatPromptTemplate.from_template(
    """Given these extracted complaint details, generate an internal support ticket.
Return ONLY valid JSON with these keys:
- ticket_id (format: HLTH-XXXX where X is a random digit)
- priority (map severity: critical→P1, high→P2, medium→P3, low→P4)
- department (map issue_type: billing→Finance, claim_denial→Claims, service_delay→Operations, coverage_question→Underwriting, other→General)
- sla_hours (P1=4, P2=8, P3=24, P4=48)
- customer_name
- summary

Extracted details: {extracted}"""
)

transform_chain = transform_prompt | llm_openai | JsonOutputParser()

# --- Step 3: Generate acknowledgment email ---
email_prompt = ChatPromptTemplate.from_template(
    """Write a brief, empathetic acknowledgment email to the customer based on this support ticket.
Include the ticket ID, expected resolution time, and assigned department.
Keep it professional and under 100 words.

Ticket details: {ticket}"""
)

email_chain = email_prompt | llm_openai | StrOutputParser()

In [14]:
# --- Run the full pipeline on a sample complaint ---
sample_complaint = """Dear Sir/Madam,
I am Rajesh Sharma and I have been a loyal customer of your HealthGuard Premium plan for 3 years.
Last month, my wife was hospitalized for an emergency surgery. I submitted the claim (Ref: CLM-78234)
two weeks ago but have received NO response. When I called your helpline, I was put on hold for
45 minutes and then disconnected. This is absolutely unacceptable for a premium plan customer.
The hospital is now sending me payment reminders. Please resolve this URGENTLY.
Regards, Rajesh Sharma"""

# Step 1: Extract
print("📋 Step 1 — Extracting details...")
extracted = extract_chain.invoke({"complaint": sample_complaint})
print(f"Extracted: {extracted}")

# Step 2: Transform
print("\n🔄 Step 2 — Creating internal ticket...")
ticket = transform_chain.invoke({"extracted": str(extracted)})
print(f"Ticket: {ticket}")

# Step 3: Generate email
print("\n📧 Step 3 — Generating acknowledgment email...")
email = email_chain.invoke({"ticket": str(ticket)})
print(f"Email:\n{email}")

📋 Step 1 — Extracting details...


Extracted: {'customer_name': 'Rajesh Sharma', 'issue_type': 'service_delay', 'severity': 'critical', 'product_mentioned': 'HealthGuard Premium plan', 'brief_summary': "Rajesh Sharma experienced a critical delay in claim processing and poor customer service for his HealthGuard Premium plan after his wife's emergency surgery."}

🔄 Step 2 — Creating internal ticket...


Ticket: {'ticket_id': 'HLTH-4827', 'priority': 'P1', 'department': 'Operations', 'sla_hours': 4, 'customer_name': 'Rajesh Sharma', 'summary': "Rajesh Sharma experienced a critical delay in claim processing and poor customer service for his HealthGuard Premium plan after his wife's emergency surgery."}

📧 Step 3 — Generating acknowledgment email...


Email:
Subject: Acknowledgment of Your Support Ticket HLTH-4827

Dear Mr. Sharma,

Thank you for reaching out regarding the delay and service issues with your HealthGuard Premium plan. We understand the urgency and sincerely empathize with your situation. Your ticket (ID: HLTH-4827) has been assigned to our Operations team, and we aim to resolve this within the next 4 hours.

We appreciate your patience and will keep you updated promptly.

Best regards,  
[Your Name]  
Customer Support Team


In [15]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def run_full_pipeline(complaint_text):
    """End-to-end complaint processing: extract → transform → generate email."""
    # Step 1
    extracted = extract_chain.invoke({"complaint": complaint_text})
    # Step 2
    ticket = transform_chain.invoke({"extracted": str(extracted)})
    # Step 3
    email = email_chain.invoke({"ticket": str(ticket)})

    return {
        "extracted": extracted,
        "ticket": ticket,
        "email": email
    }

# Wrap as a Runnable so it integrates with LCEL ecosystem
pipeline = RunnableLambda(run_full_pipeline)

# Test with a different complaint
new_complaint = """Hi, I'm Meena Patel. I noticed I was charged ₹2,400 for a wellness checkup
that should be covered under my Silver Health plan's preventive care benefits.
Can you please look into this billing error? My policy number is SH-90211. Thanks."""

result = pipeline.invoke(new_complaint)

print("🎯 Full pipeline result:")
print(f"\n📋 Extracted: {result['extracted']}")
print(f"\n🎫 Ticket: {result['ticket']}")
print(f"\n📧 Email:\n{result['email']}")

🎯 Full pipeline result:

📋 Extracted: {'customer_name': 'Meena Patel', 'issue_type': 'billing', 'severity': 'medium', 'product_mentioned': 'Silver Health plan', 'brief_summary': 'Customer was incorrectly charged ₹2,400 for a wellness checkup that should be covered under preventive care benefits.'}

🎫 Ticket: {'ticket_id': 'HLTH-4827', 'priority': 'P3', 'department': 'Finance', 'sla_hours': 24, 'customer_name': 'Meena Patel', 'summary': 'Customer was incorrectly charged ₹2,400 for a wellness checkup that should be covered under preventive care benefits.'}

📧 Email:
Subject: Acknowledgment of Your Support Ticket HLTH-4827

Dear Meena Patel,

Thank you for reaching out regarding the incorrect charge of ₹2,400 for your wellness checkup. We understand your concern and have assigned your case to our Finance department. Your ticket ID is HLTH-4827, and we aim to resolve this within 24 hours.

We appreciate your patience and will keep you updated on the progress.

Best regards,  
[Your Company

---

## 8. Parallel Branches with `RunnableParallel`

When chain branches are independent (e.g., classification + sentiment + summary on the same input), run them concurrently. `RunnableParallel` schedules them with `asyncio.gather` under the hood — typical 2–4× wall-clock speedup for I/O-bound LLM calls.


In [16]:
from langchain_core.runnables import RunnableParallel

product_description = """NutriVita Pro — Advanced probiotic supplement with 50 billion CFU,
15 clinically studied strains, delayed-release capsules for maximum gut absorption.
Supports digestive health, immune function, and mental clarity. Doctor-recommended.
Now 30% off for new customers. 90-day money-back guarantee."""

# Three parallel analysis chains
marketing_chain = ChatPromptTemplate.from_template(
    "Rate this product description's marketing appeal (1-10) and suggest one improvement. "
    "Be brief.\n\nDescription: {text}"
) | llm_openai | StrOutputParser()

compliance_chain = ChatPromptTemplate.from_template(
    "Check this product description for potential regulatory compliance issues "
    "(FDA, FTC advertising rules). List any concerns briefly.\n\nDescription: {text}"
) | llm_openai | StrOutputParser()

sentiment_chain = ChatPromptTemplate.from_template(
    "Analyze the tone and sentiment of this product description. "
    "Is it trustworthy or hype-heavy? One paragraph.\n\nDescription: {text}"
) | llm_groq | StrOutputParser()  # Using Groq for variety

# Run all three in parallel
parallel_analysis = RunnableParallel(
    marketing=marketing_chain,
    compliance=compliance_chain,
    sentiment=sentiment_chain
)

results = parallel_analysis.invoke({"text": product_description})

print("🔀 Parallel Analysis Results:\n")
for key, value in results.items():
    print(f"{'─' * 50}")
    print(f"📊 {key.upper()}")
    print(f"{'─' * 50}")
    print(value)
    print()

🔀 Parallel Analysis Results:

──────────────────────────────────────────────────
📊 MARKETING
──────────────────────────────────────────────────
Rating: 8

Improvement: Add a relatable benefit-driven sentence to connect emotionally, e.g., "Feel energized and balanced every day with NutriVita Pro."

──────────────────────────────────────────────────
📊 COMPLIANCE
──────────────────────────────────────────────────
Potential regulatory compliance issues:

1. **Disease Claim/Health Claim**:  
   - "Supports digestive health, immune function, and mental clarity" could be seen as structure/function claims. These must be truthful, not misleading, and substantiated. The claim "mental clarity" is more ambiguous and might require strong evidence or risk being considered an unapproved drug claim.

2. **"Doctor-recommended"**:  
   - This claim must be truthful and substantiated. The company should have documented evidence that doctors recommend the product.

3. **"15 clinically studied strains"**: 

In [17]:
import time

# Shared prompt — works with any LLM
summary_prompt = ChatPromptTemplate.from_template(
    """Summarize this financial news in exactly 2 sentences for a retail investor audience.

News: {news}"""
)

news_text = """The Reserve Bank of India kept the repo rate unchanged at 6.5% for the eighth
consecutive time, citing persistent food inflation concerns. However, the central bank revised
its GDP growth forecast for FY25 upward to 7.2% from the earlier 7.0%, signaling confidence
in India's economic momentum despite global uncertainties."""

# Test with both providers
for name, provider_model in [("OpenAI (gpt-4.1-mini)", llm_openai), ("Groq (Qwen3.8 27B)", llm_groq)]:
    chain = summary_prompt | provider_model | StrOutputParser()
    start = time.time()
    result = chain.invoke({"news": news_text})
    latency = round(time.time() - start, 2)
    print(f"\n🤖 {name} ({latency}s):")
    print(f"   {result}")


🤖 OpenAI (gpt-4.1-mini) (1.53s):
   The Reserve Bank of India has kept the repo rate steady at 6.5% to manage ongoing food inflation, providing stability for borrowers and investors. At the same time, it raised the GDP growth forecast for FY25 to 7.2%, showing strong confidence in India’s economic growth despite global challenges.



🤖 Groq (Qwen3.8 27B) (0.22s):
   The Reserve Bank of India has held its benchmark interest rate steady at 6.5% for the eighth month in a row due to ongoing food inflation pressures. This decision comes alongside a boost to economic optimism, as the central bank raised its GDP growth forecast for the coming year to 7.2%.


---

## 9. Conclusion & Key Takeaways

| Concept | What you should walk away with |
|---|---|
| **LCEL pipe** | `prompt \| llm \| parser` is a `RunnableSequence` — fully composable, streamable, batchable |
| **Memory** | One list of messages per session, injected through `MessagesPlaceholder` — the pattern `RunnableWithMessageHistory` used to wrap, whose history classes are deprecated since langchain-core 1.6.4 |
| **Per-session state** | History is keyed by `session_id` — same chain handles N parallel conversations |
| **`.astream_events()`** | Yields all internal events — for streaming UIs that show progress, not just final tokens |
| **`RunnableLambda`** | Escape hatch from LCEL into ordinary Python without breaking the pipe |
| **`RunnableParallel`** | Independent branches run concurrently — measurable latency win for multi-aspect analysis |

## 10. Stretch Exercise (Optional)
1. **Persistent history.** Save each session's messages with `messages_to_dict` and reload them with `messages_from_dict` (Lab 4.2 does exactly this) — verify history survives a kernel restart.
2. **Token budget guard.** Trim the history to the last 10 turns with `trim_messages` before it goes into the prompt.
3. **Streaming UI mock.** Use `.astream_events()` to print a status line ("⏳ thinking…") on `on_chat_model_start` and clear it on `on_chat_model_stream`.
4. **Memory + parallel.** Fan out the user's message into 3 parallel branches (sentiment, intent, summary) using `RunnableParallel`, then store the synthesized result in conversation history.
5. **Per-user system prompt.** Add a `user_role` argument to `chat_with_memory(...)` and have the system prompt vary by role (`compliance_officer`, `customer`, `analyst`).

---

**Next Lab:** Lab 2.2 — LLM Robustness: Retries, Fallback & Hallucination Signals 🛡️
